**Laboratorio de Métodos Cuantitativos Aplicados a la Gestión**

---

# **Clase 18 - Aplicaciones económicas de la integración**

## Complemento

Este notebook se complementa con la presentación: **Integrales.pdf**

Te recomendamos leer el PDF para trabajar con este notebook y tener una mejor comprensión de los conceptos teóricos.

## ¿Qué vamos a hacer en esta clase?

En las dos clases anteriores integramos **funciones**: teníamos una fórmula, aplicábamos `sp.integrate`
y salía el resultado exacto.

**Hoy nos pasa lo que pasa en el trabajo real: no hay fórmula. Hay datos.**

Nadie te va a dar la función de la inflación argentina ni la de las ventas de tu empresa. Te van a dar
una planilla con números. Y aun así vas a tener que calcular acumulados, áreas y excedentes.

| Parte | Tema | Herramienta |
|---|---|---|
| **A** | Repaso: de marginal a total | `sp.integrate` |
| **B** | Cuando no hay fórmula: integración numérica | Regla del trapecio |
| **C** | Caso 1: inflación acumulada | `np.trapezoid` |
| **D** | Caso 2: ingreso acumulado de un negocio | Datos semanales |
| **E** | Caso 3: desigualdad — curva de Lorenz y Gini | La integral como área |
| **F** | Caso 4: excedente del consumidor con datos reales | Volvemos al mercado |

> **La idea de fondo:** una integral **siempre** es un área bajo una curva. Si tenés la fórmula, la
> calculás exacta. Si solo tenés puntos, la **aproximás**. Y para decidir, la aproximación alcanza.

In [ ]:
import numpy as np
import pandas as pd
import sympy as sp
import matplotlib.pyplot as plt

pd.set_option("display.float_format", "{:,.2f}".format)
URL = "https://raw.githubusercontent.com/Datso653/Laboratorio-de-metodos-Cuantitativos-Aplicados-a-la-gestion/main/DF/"

# numpy cambió el nombre de la función en la versión 2.0. Esta línea funciona con las dos.
trapecio = getattr(np, "trapezoid", None) or np.trapz

---
# 🔄 Parte A — Repaso rápido: de marginal a total

Lo que ya sabemos: si conocemos la función **marginal**, integrando llegamos a la **total**.

$$C(q) = \int C'(q)\, dq + C_0$$

Un ejemplo relámpago para calentar:

In [ ]:
q = sp.symbols('q', positive=True)

costo_marginal = 3*q**2 + 20*q + 150      # C'(q)
costo_fijo = 5000                          # la constante de integración: el costo de no producir nada

costo_total = sp.integrate(costo_marginal, q) + costo_fijo

print("Costo marginal:", costo_marginal)
print("Costo total   :", costo_total)
print(f"\nProducir 50 unidades cuesta: $ {float(costo_total.subs(q, 50)):,.0f}")

Perfecto. **Pero esto exigió que alguien nos diera la fórmula de $C'(q)$.**

En una empresa nadie tiene esa fórmula. Lo que hay es una planilla con el costo de cada mes.
¿Cómo integramos eso?

---
# 📐 Parte B — Integración numérica: la regla del trapecio

La integral es el **área bajo la curva**. Si en vez de una curva tenemos puntos sueltos, aproximamos
el área uniendo los puntos con rectas y sumando el área de los **trapecios** que quedan.

```
   y                                  El área de cada trapecio es:
   │        ●───────●
   │       ╱│░░░░░░░│╲                  (base_menor + base_mayor)
   │  ●───╱ │░░░░░░░│ ╲───●             ──────────────────────── × ancho
   │ ╱│░░░░░│░░░░░░░│░░░░░│╲                       2
   │╱ │░░░░░│░░░░░░░│░░░░░│ ╲
   └──┴─────┴───────┴─────┴──── x       y sumamos todos.
```

Cuantos más puntos tengamos, mejor la aproximación. Comprobémoslo con una función que sabemos integrar
a mano, para ver **cuánto error** cometemos.

In [ ]:
# Función de prueba: f(x) = x², cuya integral entre 0 y 3 sabemos que es 3³/3 = 9 exacto
def f(x):
    return x ** 2

exacto = 9.0

print(f"{'Puntos':>8} | {'Aproximación':>13} | {'Error':>9}")
print("-" * 36)
for n in [4, 11, 51, 501]:
    x = np.linspace(0, 3, n)          # linspace: n puntos igualmente espaciados entre 0 y 3
    y = f(x)
    aprox = trapecio(y, x)            # regla del trapecio
    print(f"{n:>8} | {aprox:>13.5f} | {aprox - exacto:>9.5f}")

print(f"\nValor exacto: {exacto}")

Con solo 4 puntos el error es del 5%; con 501 puntos es despreciable.

> 📌 **Lo importante:** el error **siempre** existe, pero se achica rápido. Para tomar una decisión de
> gestión, un error del 0,01% no cambia nada. La integración numérica es más que suficiente.

In [ ]:
# Visualizamos qué está haciendo el método
x_fino = np.linspace(0, 3, 200)
x_pocos = np.linspace(0, 3, 5)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x_fino, f(x_fino), color="#243b5e", linewidth=2, label="f(x) = x² (la curva real)")
ax.fill_between(x_pocos, f(x_pocos), alpha=0.35, color="#e07b39",
                label="Trapecios (la aproximación)")
ax.plot(x_pocos, f(x_pocos), "o-", color="#e07b39", markersize=7)
ax.set_title("La regla del trapecio con 5 puntos", loc="left", fontweight="bold")
ax.set_xlabel("x"); ax.set_ylabel("f(x)")
ax.legend()
plt.tight_layout()
plt.show()

Se ve clarito: los trapecios **sobreestiman** el área porque la curva es convexa y las rectas van por
arriba. Por eso el error nos dio positivo en la tabla.

---
# 💸 Parte C — Caso 1: la inflación acumulada

Este es **el** ejemplo argentino de integración. El INDEC publica la inflación **mensual**: eso es una
función marginal, la velocidad a la que suben los precios. Lo que a todos nos importa es la inflación
**acumulada**: la función total.

Vamos a trabajar con la inflación mensual de 2023, un año que todos tenemos fresco.

In [ ]:
# Inflación mensual de 2023 en Argentina, en % (fuente: INDEC, IPC Nacional)
meses = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
         "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
inflacion_mensual = np.array([6.0, 6.6, 7.7, 8.4, 7.8, 6.0,
                              6.3, 12.4, 12.7, 8.3, 12.8, 25.5])

ipc = pd.DataFrame({"mes": meses, "inflacion_mensual": inflacion_mensual})
ipc

### ⚠️ La trampa número uno: la inflación NO se suma

La tentación es sumar los 12 números. **Está mal**, y el error no es chico.

La razón: cada mes los precios suben sobre el nivel **ya aumentado** del mes anterior. Es interés
compuesto. La fórmula correcta multiplica los factores:

$$\text{acumulada} = \left[\prod_{i=1}^{12}\left(1 + \frac{\pi_i}{100}\right) - 1\right] \times 100$$

In [ ]:
suma_ingenua = inflacion_mensual.sum()

factores = 1 + inflacion_mensual / 100        # cada mes multiplica por su factor
acumulada = (np.prod(factores) - 1) * 100     # prod: multiplica todos los elementos

print(f"Sumando (MAL)      : {suma_ingenua:>6.1f} %")
print(f"Componiendo (BIEN) : {acumulada:>6.1f} %")
print(f"\nLa diferencia es de {acumulada - suma_ingenua:.0f} puntos porcentuales.")

**91 puntos de diferencia.** Ese es el costo de sumar cuando había que multiplicar.

### ¿Y dónde está la integral acá?

En el **índice de precios**. Si tomamos logaritmos, el producto se convierte en suma:

$$\ln\left(\frac{P_{final}}{P_{inicial}}\right) = \sum_i \ln(1 + \pi_i) \;\; \approx \;\; \int_0^{12} \pi(t)\, dt$$

La inflación acumulada **es el área bajo la curva de la inflación mensual**, medida en logaritmos.
Eso es exactamente una integral.

In [ ]:
# El índice de precios: cómo evoluciona el nivel general partiendo de 100
ipc["indice"] = 100 * np.cumprod(factores)      # cumprod: producto acumulado mes a mes
ipc["acumulada_%"] = ipc["indice"] - 100

ipc

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# Izquierda: la función MARGINAL (velocidad de aumento)
ax1.bar(meses, inflacion_mensual, color="#243b5e")
ax1.fill_between(range(12), inflacion_mensual, alpha=0.15, color="#e07b39")
ax1.set_title("Función MARGINAL: inflación mensual", loc="left", fontweight="bold")
ax1.set_ylabel("% mensual")

# Derecha: la función TOTAL (nivel acumulado)
ax2.plot(meses, ipc["indice"], marker="o", color="#e07b39", linewidth=2.5)
ax2.axhline(100, color="gray", linestyle=":", label="Nivel de partida")
ax2.set_title("Función TOTAL: índice de precios (base Ene = 100)", loc="left", fontweight="bold")
ax2.set_ylabel("Índice")
ax2.legend()

plt.tight_layout()
plt.show()

**Este par de gráficos resume todo el bloque de integrales:** a la izquierda la derivada (la velocidad),
a la derecha la primitiva (el acumulado). Integrar es pasar del gráfico de la izquierda al de la derecha.

---
# 🏪 Parte D — Caso 2: el ingreso acumulado de un negocio

Volvemos al supermercado de 12 semanas. Tenemos el ingreso de cada semana (marginal) y queremos el
**acumulado del período** (total). Acá sí se suma, porque los pesos de cada semana se agregan.

In [ ]:
sup = pd.read_csv(URL + "datos_supermercado_simulados.csv")

sup["ingreso"] = (sup["cant_alimentos"] * sup["precio_alimentos"] +
                  sup["cant_bebidas"]   * sup["precio_bebidas"] +
                  sup["cant_limpieza"]  * sup["precio_limpieza"])
sup["margen"] = sup["ingreso"] - sup["costos_operativos"]
sup["margen_acumulado"] = sup["margen"].cumsum()     # cumsum: suma acumulada

sup[["semana", "ingreso", "costos_operativos", "margen", "margen_acumulado"]]

In [ ]:
# Tres formas de calcular el acumulado, para comparar
suma_simple = sup["ingreso"].sum()
area_trapecio = trapecio(sup["ingreso"].values, sup["semana"].values)

print(f"Suma de las 12 semanas      : $ {suma_simple:>12,.0f}")
print(f"Área por regla del trapecio : $ {area_trapecio:>12,.0f}")
print(f"Diferencia                  : $ {suma_simple - area_trapecio:>12,.0f}")

### ¿Por qué no dan igual? (y cuál está bien)

Las dos son correctas, pero **responden preguntas distintas**:

| Método | Supone que... | Sirve para |
|---|---|---|
| **Suma** (`.sum()`) | Cada dato es un **total ya cerrado** del período | Ingresos, unidades vendidas, gastos |
| **Trapecio** | Los datos son **mediciones instantáneas** de algo continuo | Stock, temperatura, tasa de producción |

Acá el ingreso semanal ya es un total cerrado → **la suma es la correcta**. El trapecio subestima
porque le "corta" media semana en cada extremo.

> 🎯 **La pregunta que hay que hacerse siempre:** ¿mi dato es un *flujo ya acumulado* (se suma) o una
> *medición puntual de una velocidad* (se integra)? Confundirlos es un error clásico en los TP.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

ax.bar(sup["semana"], sup["margen"], color="#243b5e", alpha=0.75, label="Margen semanal")
ax2 = ax.twinx()          # twinx: un segundo eje Y del lado derecho
ax2.plot(sup["semana"], sup["margen_acumulado"], marker="o",
         color="#e07b39", linewidth=2.5, label="Margen acumulado")

ax.set_xlabel("Semana")
ax.set_ylabel("Margen semanal ($)")
ax2.set_ylabel("Margen acumulado ($)")
ax.set_title("Margen semanal (barras) vs acumulado (línea)", loc="left", fontweight="bold")
ax.legend(loc="upper left", fontsize=8)
ax2.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

---
# ⚖️ Parte E — Caso 3: medir la desigualdad con una integral

Una de las aplicaciones más elegantes de la integración en economía: **el coeficiente de Gini**.

### La curva de Lorenz

Ordenamos a la población de menor a mayor ingreso y preguntamos: *"¿qué porcentaje del ingreso total
se lleva el X% más pobre?"*

- Si todos ganaran **exactamente lo mismo**, el 20% más pobre se llevaría el 20% del ingreso, el 40%
  se llevaría el 40%... eso dibuja una **diagonal perfecta**: la *línea de igualdad*.
- En la realidad la curva se hunde por debajo de la diagonal. **Cuánto se hunde es la desigualdad.**

### El Gini

$$G = \frac{\text{área entre la diagonal y la curva}}{\text{área bajo la diagonal}} = 1 - 2\int_0^1 L(p)\, dp$$

Va de **0** (igualdad total) a **1** (una sola persona se queda con todo). **Es una integral, y la
vamos a calcular con el trapecio.**

In [ ]:
# Distribución del ingreso por deciles (10 grupos del 10% de la población cada uno).
# Cada número es el % del ingreso total que se lleva ese decil, del más pobre al más rico.
deciles = np.array([1.5, 2.8, 3.9, 5.2, 6.7, 8.5, 10.8, 13.9, 18.7, 28.0])

print("Suma de participaciones:", deciles.sum(), "%")   # tiene que dar 100

In [ ]:
# Construimos la curva de Lorenz: proporción acumulada de población vs de ingreso
p = np.linspace(0, 1, 11)                                  # 0%, 10%, ..., 100% de la población
L = np.concatenate([[0], np.cumsum(deciles) / 100])        # ingreso acumulado (arranca en 0)

lorenz = pd.DataFrame({"pob_acumulada": p, "ingreso_acumulado": L})
lorenz.round(3)

In [ ]:
# El Gini: 1 menos el doble del área bajo la curva de Lorenz
area_lorenz = trapecio(L, p)
gini = 1 - 2 * area_lorenz

print(f"Área bajo la curva de Lorenz : {area_lorenz:.4f}")
print(f"Área bajo la diagonal        : 0.5000  (es un triángulo: base × altura / 2)")
print(f"\nCoeficiente de Gini          : {gini:.4f}")
print()
print("Referencias reales:")
print("  Argentina (INDEC, 2023) ≈ 0.435")
print("  Países nórdicos         ≈ 0.25 - 0.28")
print("  Sudáfrica               ≈ 0.63  (el más alto del mundo)")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6))

ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=2, label="Igualdad perfecta")
ax.plot(p, L, marker="o", color="#243b5e", linewidth=2.5, label="Curva de Lorenz")
ax.fill_between(p, L, p, color="#e07b39", alpha=0.35,
                label=f"Desigualdad → Gini = {gini:.3f}")

ax.set_xlabel("Proporción acumulada de la población")
ax.set_ylabel("Proporción acumulada del ingreso")
ax.set_title("Curva de Lorenz", loc="left", fontweight="bold")
ax.legend(loc="upper left", fontsize=9)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

**El área naranja es el Gini.** Una integral que sale todos los años en los diarios y que define
políticas públicas.

Fijate el dato del gráfico: el 50% más pobre se lleva apenas el **20%** del ingreso, mientras que el
decil más rico solo se lleva el **28%**.

---
# 🛒 Parte F — Caso 4: excedente del consumidor con datos reales

En la clase anterior calculamos el excedente del consumidor con la fórmula de la demanda.
Ahora lo hacemos **como en la vida real**: con una tabla de datos observados.

$$EC = \int_0^{q^*} P(q)\, dq \; - \; p^* \cdot q^*$$

Es decir: el área bajo la curva de demanda, menos lo que efectivamente se pagó.

In [ ]:
# Datos observados de un estudio de mercado: a cada precio, cuánto se vendería
demanda = pd.DataFrame({
    "precio":   [100, 90, 80, 70, 60, 50, 40, 30, 20, 10],
    "cantidad": [  0, 12, 25, 39, 54, 70, 87, 105, 124, 144],
})

demanda

In [ ]:
precio_mercado = 50
cantidad_vendida = 70          # a $50 se venden 70 unidades

# Nos quedamos con el tramo relevante: de q=0 hasta la cantidad vendida
tramo = demanda[demanda["cantidad"] <= cantidad_vendida].sort_values("cantidad")

# Área total bajo la curva de demanda (lo máximo que los consumidores estaban dispuestos a pagar)
disposicion_a_pagar = trapecio(tramo["precio"].values, tramo["cantidad"].values)

# Lo que efectivamente pagaron
gasto_real = precio_mercado * cantidad_vendida

excedente = disposicion_a_pagar - gasto_real

print(f"Disposición total a pagar : $ {disposicion_a_pagar:>8,.0f}   ← área bajo la demanda")
print(f"Lo que realmente pagaron  : $ {gasto_real:>8,.0f}   ← precio × cantidad")
print(f"{'-' * 44}")
print(f"EXCEDENTE DEL CONSUMIDOR  : $ {excedente:>8,.0f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(demanda["cantidad"], demanda["precio"], marker="o",
        color="#243b5e", linewidth=2.5, label="Curva de demanda")
ax.fill_between(tramo["cantidad"], tramo["precio"], precio_mercado,
                color="#e07b39", alpha=0.4, label=f"Excedente = ${excedente:,.0f}")
ax.axhline(precio_mercado, color="gray", linestyle="--", label=f"Precio de mercado = ${precio_mercado}")
ax.axvline(cantidad_vendida, color="gray", linestyle=":", alpha=0.6)

ax.set_xlabel("Cantidad")
ax.set_ylabel("Precio ($)")
ax.set_title("Excedente del consumidor calculado con datos observados",
             loc="left", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

**El excedente es el beneficio que se llevan los consumidores** por pagar menos de lo que estaban
dispuestos a pagar. Y lo calculamos **sin conocer la fórmula de la demanda**: solo con una tabla
de 10 filas y la regla del trapecio.

> 🎯 Esto es exactamente lo que hace un regulador cuando evalúa si una fusión de empresas perjudica
> a los consumidores, o lo que hace una empresa cuando decide cuánto puede subir un precio.

---
# 📝 Ejercicios

**Ejercicio 1.** La inflación mensual de un semestre fue: 4,2 · 3,8 · 5,1 · 4,6 · 3,9 · 4,4 (en %).
Calculá la inflación acumulada del semestre. ¿Cuánto da si la sumás mal? ¿Cuántos puntos de diferencia hay?

In [ ]:
# Tu respuesta acá

**Ejercicio 2.** Con los datos del supermercado, calculá el **costo operativo acumulado** de las 12
semanas y agregá una columna de costo acumulado al DataFrame. Graficá margen acumulado contra costo acumulado.

In [ ]:
# Tu respuesta acá

**Ejercicio 3.** Una fábrica mide su ritmo de producción (unidades por hora) cada 2 horas durante una
jornada de 8 horas: `[120, 145, 160, 155, 130]` a las horas `[0, 2, 4, 6, 8]`.
¿Cuántas unidades produjo en total? **Pensá bien:** ¿esto se suma o se integra?

In [ ]:
# Tu respuesta acá

**Ejercicio 4.** Calculá el Gini de una distribución **más igualitaria**:
`[4.0, 5.5, 6.8, 8.0, 9.2, 10.5, 12.0, 13.5, 15.0, 15.5]`.
Graficá las dos curvas de Lorenz juntas y compará.

In [ ]:
# Tu respuesta acá

**Ejercicio 5.** En el caso del excedente, ¿qué pasa si el precio de mercado baja a $40?
Recalculá el excedente del consumidor. ¿Aumentó o disminuyó? ¿Por qué tiene sentido económico?

In [ ]:
# Tu respuesta acá

**Ejercicio 6 (integrador 🥇⚡🤓).** Compará la regla del trapecio con `scipy.integrate.simpson`
(regla de Simpson, que usa parábolas en vez de rectas) sobre la función $f(x)=x^3$ entre 0 y 4,
cuyo valor exacto es 64. ¿Cuál se equivoca menos y por qué te parece?

```python
from scipy.integrate import simpson
```

In [ ]:
# Tu respuesta acá

---
## 🧭 Para llevarse

| Situación | Herramienta |
|---|---|
| Tengo la **fórmula** | `sp.integrate(f, x)` → resultado exacto |
| Tengo **datos** de una velocidad o tasa | `trapecio(y, x)` → área aproximada |
| Tengo **datos** de totales ya cerrados | `.sum()` |
| Quiero la evolución acumulada | `.cumsum()` (sumas) o `.cumprod()` (tasas) |
| Tasas de crecimiento (inflación, interés) | **Multiplicar factores**, nunca sumar |

**Las tres frases para llevarse:**
1. Una integral **siempre** es un área bajo una curva.
2. Sin fórmula igual se integra: la regla del trapecio alcanza para decidir.
3. Antes de sumar, preguntate si el dato es un **total cerrado** o una **velocidad**.

> Con esta clase cerramos la Unidad 4. En la próxima arrancamos con simulación de datos.